# Networking RAG - Phase 4 Evaluation

This notebook evaluates the improved Networking RAG system developed in Phase 4. The evaluation is performed using the same test questions and RAGAS metrics used previously, allowing comparison between the baseline and improved retrieval pipelines.

In [1]:
# Install required libraries

try:

    !pip install -r requirements.txt

    print(
        "Dependencies installed successfully!"
    )

except Exception as e:

    print(
        "Installation Error:"
    )

    print(e)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.0/349.0 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

Dependencies installed successfully!


In [1]:
# Import required libraries

import os
import time
import pickle
import chromadb
import pandas as pd

from datasets import Dataset

from google import genai
from google.colab import userdata

from groq import Groq

print(
    "Libraries imported successfully!"
)

Libraries imported successfully!


In [2]:
# Import RAGAS

try:

    import ragas

    print(
        "RAGAS imported successfully!"
    )

    print(
        "Version:",
        ragas.__version__
    )

except Exception as e:

    print(e)

RAGAS imported successfully!
Version: 0.4.3


In [3]:
# Initialize Gemini

try:

    gemini_client = genai.Client(
        api_key=userdata.get(
            "GEMINI_API_KEY"
        )
    )

    print(
        "Gemini initialized!"
    )

except Exception as e:

    print(e)

Gemini initialized!


In [4]:
# Initialize Groq

try:

    groq_client = Groq(
        api_key=userdata.get(
            "GROQ_API_KEY"
        )
    )

    print(
        "Groq initialized!"
    )

except Exception as e:

    print(e)

Groq initialized!


In [5]:
# Load ChromaDB

try:

    chroma_client = chromadb.PersistentClient(
        path="./networking_chromadb_phase4"
    )

    collection = (
        chroma_client.get_or_create_collection(
            name="networking_docs_phase4"
        )
    )

    print(
        "Records:",
        collection.count()
    )

except Exception as e:

    print(e)

Records: 0


In [6]:
# Load saved chunks and embeddings

try:

    with open(
        "improved_chunks.pkl",
        "rb"
    ) as f:

        improved_chunks = pickle.load(f)

    with open(
        "improved_embeddings.pkl",
        "rb"
    ) as f:

        improved_embeddings = pickle.load(f)

    if collection.count() == 0:

        collection.add(

            ids=[
                f"chunk_{i}"
                for i in range(
                    len(improved_chunks)
                )
            ],

            documents=
            improved_chunks,

            embeddings=
            improved_embeddings
        )

    print(
        "Collection ready!"
    )

    print(
        "Records:",
        collection.count()
    )

except Exception as e:

    print(e)

Collection ready!
Records: 506


In [13]:
# Load evaluation questions

try:

    def load_evaluation_data(
        path="arrays.txt"
    ):

        globals_dict = {}

        with open(
            path,
            "r"
        ) as f:

            exec(
                f.read(),
                globals_dict
            )

        return (

            globals_dict[
                "test_questions"
            ],

            globals_dict[
                "ground_truths"
            ]

        )

    test_questions, ground_truths = (

        load_evaluation_data()

    )

    # Keep only first 5 questions

    test_questions = (
        test_questions[:5]
    )

    ground_truths = (
        ground_truths[:5]
    )

    print(
        "Questions:",
        len(test_questions)
    )

    print(
        "Ground Truths:",
        len(ground_truths)
    )

except Exception as e:

    print(
        "Loading Error:"
    )

    print(e)

Questions: 5
Ground Truths: 5


In [9]:
# Install BM25 library

try:

    !pip install -q rank-bm25

    print(
        "BM25 library installed successfully!"
    )

except Exception as e:

    print(
        "Installation Error:"
    )

    print(e)

BM25 library installed successfully!


In [14]:
# Create BM25 index

try:

    from rank_bm25 import BM25Okapi

    tokenized_chunks = [

        chunk.lower().split()

        for chunk in improved_chunks

    ]

    bm25 = BM25Okapi(
        tokenized_chunks
    )

    print(
        "BM25 index created successfully!"
    )

    print(
        "Indexed Chunks:",
        len(tokenized_chunks)
    )

except Exception as e:

    print(e)

BM25 index created successfully!
Indexed Chunks: 506


In [15]:
# RAG Query Function

try:

    def ask_network_question(
        question
    ):

        query_response = (
            gemini_client.models.embed_content(
                model="models/gemini-embedding-001",
                contents=question
            )
        )

        query_embedding = (
            query_response.embeddings[0].values
        )

        vector_results = (
            collection.query(
                query_embeddings=[
                    query_embedding
                ],
                n_results=5
            )
        )

        vector_chunks = (
            vector_results["documents"][0]
        )

        tokenized_query = (
            question.lower().split()
        )

        bm25_scores = (
            bm25.get_scores(
                tokenized_query
            )
        )

        bm25_indices = (
            sorted(
                range(
                    len(bm25_scores)
                ),
                key=lambda i:
                bm25_scores[i],
                reverse=True
            )[:5]
        )

        bm25_chunks = [

            improved_chunks[i]

            for i in bm25_indices

        ]

        retrieved_chunks = []

        for chunk in (
            vector_chunks +
            bm25_chunks
        ):

            if chunk not in retrieved_chunks:

                retrieved_chunks.append(
                    chunk
                )

        retrieved_chunks = (
            retrieved_chunks[:5]
        )

        context = "\n\n".join(
            retrieved_chunks
        )

        prompt = f"""
You are a networking assistant.

Answer only using the provided context.

Context:
{context}

Question:
{question}
"""

        response = (
            groq_client.chat.completions.create(
                model=
                "llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )
        )

        return {

            "answer":
            response.choices[0]
            .message.content,

            "contexts":
            retrieved_chunks
        }

    print(
        "Function created successfully!"
    )

except Exception as e:

    print(e)

Function created successfully!


In [16]:
# Test hybrid search

try:

    result = (
        ask_network_question(
            "What is DNS?"
        )
    )

    print(
        result["answer"]
    )

except Exception as e:

    print(e)

The Domain Name System (DNS) is a hierarchical and distributed name service that provides a naming system for computers, services, and other resources on the Internet or other Internet Protocol (IP) networks. It associates various information with domain names and translates readily memorized domain names to the numerical IP addresses needed for locating and identifying computer services and devices.


In [17]:
# Generate answers for evaluation questions

try:

    generated_answers = []

    retrieved_contexts = []

    for i, question in enumerate(
        test_questions
    ):

        print(
            f"Processing {i+1}/{len(test_questions)}"
        )

        result = (
            ask_network_question(
                question
            )
        )

        generated_answers.append(
            result["answer"]
        )

        retrieved_contexts.append(
            result["contexts"]
        )

        if (
            (i + 1) % 2 == 0
            and
            (i + 1) < len(
                test_questions
            )
        ):

            print(
                "Rate limit protection activated. Waiting 30 seconds..."
            )

            time.sleep(30)

    print(
        "Generation completed!"
    )

except Exception as e:

    print(
        "Generation Error:"
    )

    print(e)

Processing 1/5
Processing 2/5
Rate limit protection activated. Waiting 30 seconds...
Processing 3/5
Processing 4/5
Rate limit protection activated. Waiting 30 seconds...
Processing 5/5
Generation completed!


In [18]:
# Create Evaluation Dataset

try:

    evaluation_dataset = (
        Dataset.from_dict(
            {
                "question":
                test_questions,

                "answer":
                generated_answers,

                "contexts":
                retrieved_contexts,

                "ground_truth":
                ground_truths
            }
        )
    )

    print(
        evaluation_dataset
    )

except Exception as e:

    print(
        "Dataset Error:"
    )

    print(e)

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})


In [19]:
# Configure evaluator models

try:

    from langchain_groq import ChatGroq

    from langchain_google_genai import (
        GoogleGenerativeAIEmbeddings
    )

    evaluator_llm = (
        ChatGroq(
            model=
            "llama-3.3-70b-versatile",

            api_key=
            userdata.get(
                "GROQ_API_KEY"
            )
        )
    )

    evaluator_embeddings = (
        GoogleGenerativeAIEmbeddings(
            model=
            "models/gemini-embedding-001",

            google_api_key=
            userdata.get(
                "GEMINI_API_KEY"
            )
        )
    )

    print(
        "Evaluator configured successfully!"
    )

except Exception as e:

    print(
        "Configuration Error:"
    )

    print(e)

Evaluator configured successfully!


In [20]:
# Import RAGAS metrics

try:

    from ragas import evaluate

    from ragas.metrics import (
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    )

    print(
        "Metrics imported successfully!"
    )

except Exception as e:

    print(
        "Import Error:"
    )

    print(e)

Metrics imported successfully!


/tmp/ipykernel_1065/4155784304.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_1065/4155784304.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_1065/4155784304.py:7: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_1065/4155784304.py:7: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed

In [23]:
from ragas.run_config import RunConfig

run_config = RunConfig(
    timeout=300
)

In [24]:
# Evaluate improved RAG pipeline

try:

    results = evaluate(

        evaluation_dataset,

        metrics=[

            faithfulness,

            answer_relevancy,

            context_precision,

            context_recall

        ],

        llm=evaluator_llm,

        embeddings=
        evaluator_embeddings,
        run_config=run_config
    )

    print(
        "Evaluation completed successfully!"
    )

    print(
        results
    )

except Exception as e:

    print(
        "Evaluation Error:"
    )

    print(e)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[19]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kt3chpnje1gv4mh7h61dgbea` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98191, Requested 1990. Please try again in 2m36.384s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})


Evaluation completed successfully!
{'faithfulness': 1.0000, 'answer_relevancy': 0.8040, 'context_precision': 0.9258, 'context_recall': 1.0000}


In [25]:
# Save evaluation results

try:

    results_df = (
        results.to_pandas()
    )

    print(
        results_df
    )

    results_df.to_csv(

        "phase4_ragas_results.csv",

        index=False

    )

    print(
        "Results saved successfully!"
    )

except Exception as e:

    print(e)

                     user_input  \
0                  What is TCP?   
1                  What is UDP?   
2                  What is DNS?   
3  What is the purpose of HTTP?   
4        What is the OSI model?   

                                  retrieved_contexts  \
0  [TCP veto, The Transmission Control Protocol (...   
1  [In computer networking, the User Datagram Pro...   
2  [The Domain Name System (DNS) is a hierarchica...   
3  [HTTP (Hypertext Transfer Protocol) is an appl...   
4  [The Open Systems Interconnection (OSI) model ...   

                                            response  \
0  The Transmission Control Protocol (TCP) is one...   
1  UDP is the User Datagram Protocol, a connectio...   
2  The Domain Name System (DNS) is a hierarchical...   
3  The purpose of HTTP (Hypertext Transfer Protoc...   
4  The OSI model is a reference model developed b...   

                                           reference  faithfulness  \
0  TCP is a reliable connection-oriented tran